In [2]:
import yaml
import joblib
import re
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_anthropic import ChatAnthropic
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import PromptTemplate
from prompt_templates import RAG_PROMPT_PART1

with open('secret.key') as key_file:
    api_key = key_file.readline().strip('\n').strip()

In [3]:
with open('config.yaml', 'rb') as yaml_file:
    config=yaml.safe_load(yaml_file)

In [4]:
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m')

In [5]:
try:
    chroma_vectorstore = Chroma(
        collection_name=config['collection_name'],
        persist_directory=config['chroma_dir'],
        embedding_function=embeddings,
        create_collection_if_not_exists=False
    )
    dense_retriever = chroma_vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': config['retrieval_k']},
    )
    with open(config['bm25_index_path'], 'rb') as bm25_file:
        sparse_retriever = joblib.load(bm25_file)

    hybrid_retriever = EnsembleRetriever(
        retrievers=[dense_retriever, sparse_retriever],
        weights=[config['vector_search_weightage'], (1-config['vector_search_weightage'])],
    )
except:
    print('[Error] Vector database or bm25 index missing, please run part1_1 script first')

In [6]:
llm = ChatAnthropic(
    # model_name='claude-haiku-4-5-20251001',
    model_name='claude-sonnet-5',
    # temperature=0.0,
    # top_k=10,
    # top_p=0.5,
    max_tokens=4096,
    timeout=30,
    max_retries=2,
    anthropic_api_key=api_key
)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain-anthropic': '1.4.8'}} output_version=None model='claude-sonnet-5' max_tokens=4096 default_request_timeout=30.0 anthropic_api_url='https://api.anthropic.com' anthropic_api_key=SecretStr('**********') anthropic_proxy=None model_kwargs={}


In [7]:
retrieval_prompt = PromptTemplate(
    input_variables=['context', 'input'],
    template_format='f-string',
    template=RAG_PROMPT_PART1
)

combine_chain = create_stuff_documents_chain(
    llm=llm, 
    prompt=retrieval_prompt
)

retrieval_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=combine_chain
)

In [8]:
def extract_answer(llm_answer_text, answer_key):
    final_answer = ''

    result_text = re.search(r'<result>\s*(\{.*?\})\s*</result>', llm_answer_text, flags=re.S)
    if not result_text:
        final_answer += '[Error] LLM did not return in <result> tags as prompted\n'
        final_answer += llm_answer_text
    else:
        try:
            json_dict = json.loads(result_text.group(1))
            if json_dict.get(answer_key) is not None:
                final_answer = f'Answer: {json_dict[answer_key]}'
            else:
                final_answer += "[Error] '{answer_key}' key not found in JSON payload\n"
                final_answer += llm_answer_text
        except json.JSONDecodeError:
            final_answer += '[Error] LLM did not return answer in JSON format as prompted\n'
            final_answer += llm_answer_text
    return final_answer

In [9]:
query = 'what is the Amount of Corporate Income Tax in 2024'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The Estimated FY2024 Corporate Income Tax amount is $28.03 billion (or $28,029 million), as reported in Table 2.1 'Budget for FY2024' (page 16) and Table 3.2a 'Revenue Collections for FY2018 to FY2024 ($ million)' (page 26).
[Number] Answer: 28.03


In [10]:
query = 'what is the Amount of Corporate Income Tax in 2023'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The Corporate Income Tax amount for FY2023 (Revised) was $28.38 billion (equivalently 28,380 $ million), as reported in Table 1.1 'Fiscal Position in FY2022 and FY2023' (page 8), Table 3.2a 'Revenue Collections for FY2018 to FY2024 ($ million)' (page 26), and Table 2.1 'Budget for FY2024' (page 16).
[Number] Answer: 28.38


In [11]:
query = 'what is the YOY percentage difference of Corp Income Tax in 2024'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: According to Table 2.1 'Budget for FY2024' (page 16), Corporate Income Tax collections are estimated to decrease by 1.2% year-over-year in FY2024, from $28.38 billion (Revised FY2023) to $28.03 billion (Estimated FY2024), a change of -$0.35 billion.
[Number] Answer: -1.2


In [12]:
query = 'what is the YOY percentage difference of Corp Income Tax in 2023'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The YoY percentage difference for Corporate Income Tax in FY2023 (Revised FY2023 compared to Actual FY2022) is 23.0%, with collections rising from $23.07 billion in FY2022 to $28.38 billion in FY2023 (Table 1.1, page 8).
[Number] Answer: 23.0


In [13]:
query = 'what is the Total amount of top ups in 2024'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The Total amount of Top-ups to Endowment and Trust Funds for FY2024 is $20,352 million (approximately $20.4 billion), according to Table 2.4 'Top-ups to Endowment and Trust Funds in FY2024' (page 20). This is also referenced in Section 2.1 (page 13) as 'Top-ups to Endowment and Trust Funds of $20.4 billion.'
[Number] Answer: 20352.0


In [14]:
query = 'what is the Total amount of top ups in 2023'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The total amount of Top-ups to Endowment and Trust Funds for Revised FY2023 was $24.32 billion (equivalently $24,320 million), as reported in Table 1.1 'Fiscal Position in FY2022 and FY2023' (page 8, row 27) and confirmed by the Sub-Total in Table 1.2 'Summary of Revised FY2023 Special Transfers' (page 11, row 19: $24,320 million). This is also referenced in the narrative text on page 5 stating 'Top-ups to Endowment and Trust Funds of $24.3 billion'.
[Number] Answer: 24.32


In [15]:
query = 'what is the List of taxes mentioned in the Operating Revenue section'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
list_answer = extract_answer(rag_chain_output['answer'], 'list_answer')

# TODO custom prompt for other taxes?
print(f'[Full] {full_answer}')
print(f'[List] {list_answer}')

[Full] Answer: The Operating Revenue sections (1.2 for FY2023 Revised, page 11, and 2.2 for FY2024 Estimated, page 13) mention the following taxes: Corporate Income Tax, Other Taxes (including Foreign Worker Levy, Water Conservation Tax, Land Betterment Charge, and Annual Tonnage Tax), Personal Income Tax, Assets Taxes, Betting Taxes, Goods and Services Tax, Motor Vehicle Taxes, and Casino Taxes.
[List] Answer: ['Corporate Income Tax', 'Other Taxes', 'Personal Income Tax', 'Assets Taxes', 'Betting Taxes', 'Goods and Services Tax', 'Motor Vehicle Taxes', 'Casino Taxes']


In [16]:
query = 'what is the latest Actual Fiscal Position in billions'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The latest Actual (Revised) Overall Fiscal Position is for FY2023, at $(3.57) billion, a deficit, as reported in Table 1.1 'Fiscal Position in FY2022 and FY2023' (page 8, row 42 - OVERALL FISCAL POSITION) and corroborated by Table 3.1a 'Overall Fiscal Position for FY2018 to FY2024' (page 24, row 19), which shows FY2023 (Revised) = $(3,571) million.
[Number] Answer: -3.57


In [17]:
query = 'what is the latest Fiscal Position in billions'

rag_chain_output = retrieval_chain.invoke({'input': query})
full_answer = extract_answer(rag_chain_output['answer'], 'full_answer')
number_answer = extract_answer(rag_chain_output['answer'], 'number_answer')

print(f'[Full] {full_answer}')
print(f'[Number] {number_answer}')

[Full] Answer: The latest Fiscal Position is the Estimated FY2024 Overall Fiscal Position of $0.78 billion (a surplus, approximately 0.1% of GDP), as reported in Table 2.1: Budget for FY2024 (page 16) and confirmed in the narrative text on page 13, which states: 'the estimated Overall Fiscal Position for FY2024 is a surplus of $0.8 billion (0.1% of GDP).' This follows the Revised FY2023 Overall Fiscal Position of $(3.57) billion (a deficit), per Table 2.1 and Table 1.1 (page 8).
[Number] Answer: 0.78
